# 05 — Länderklassifikation (Gruppierung für Forschungsfrage 3)

Dieses Notebook bereitet die **Metadaten-Variablen** zur Gruppierung der 119 Studienländer vor.
Es greift ausschließlich **lesend** auf bestehende Dateien zu und schreibt neue Übersichts-/
Ergebnisdateien; die bereinigte Datenbank aus Notebook 04 bleibt unverändert.

**Geplante Merkmale (je Land × Jahr):**
1. **IMF-Ländergruppe** (Advanced Economy vs. Emerging and Developing Economies), je Jahr aus dem
   *World Economic Outlook* (WEO).
2. **Lambda/BIP** nach **Jones & Klenow** (*Beyond GDP*, Version 5.0, Querschnitt 2007) — das
   Verhältnis der konsumäquivalenten Wohlfahrt λ zum BIP pro Kopf y (beide zu USA = 100 indexiert).
3. Bewertung, ob Lambda/BIP im **globalen** Vergleich über-/unterdurchschnittlich ist.
4. Bewertung, ob Lambda/BIP im Vergleich **innerhalb der eigenen IMF-Gruppe** über-/unterdurch-
   schnittlich ist (je Jahr).

> **Umfang dieses Schritts:** Einlesen der beiden Datenquellen und eine **Abdeckungs-Übersicht**
> (für welche Länder liegen Lambda/BIP- bzw. WEO-Daten vor, und für wie viele Jahre). Die
> eigentliche Konstruktion der vier Merkmale (1–4) folgt in einem späteren Schritt.

## 0 — Setup

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Pfade (relativ zum /code-Verzeichnis)
ANALYSIS_DIR = Path('Created Files for Analysis')
PULLED       = Path('..') / 'Pulled Databases'
JK_PATH      = PULLED / 'Jones_and_Klenow_Database' / 'BeyondGDP500.xls'
WEO_PATH     = PULLED / 'WEO country classification.xlsx'

# Analyserahmen aus config.json (Notebook 01)
with open(ANALYSIS_DIR / 'config.json', 'r', encoding='utf-8') as f:
    config = json.load(f)

INCLUDED_COUNTRIES = sorted(config['included_countries'])  # 119 ISO3
YEAR_START         = config['year_start']                  # 2007
YEAR_END           = config['year_end']                    # 2022
N_YEARS            = config['n_years']                      # 16
N_COUNTRIES        = config['n_countries']                 # 119

# ISO3 -> Ländername
country_list = pd.read_excel(ANALYSIS_DIR / 'country_list.xlsx')
ISO3_TO_NAME = dict(zip(country_list['ISO3'], country_list['Country_Name']))
NAME_TO_ISO3 = {n.strip(): i for i, n in ISO3_TO_NAME.items()}

print(f'Studienländer:   {N_COUNTRIES}')
print(f'Studienzeitraum: {YEAR_START}-{YEAR_END} ({N_YEARS} Jahre)')
print(f'J&K-Quelle:      {JK_PATH.resolve().name}')
print(f'WEO-Quelle:      {WEO_PATH.resolve().name}')

Studienländer:   119
Studienzeitraum: 2007-2022 (16 Jahre)
J&K-Quelle:      BeyondGDP500.xls
WEO-Quelle:      WEO country classification.xlsx


## 1 — Datenquellen einlesen

### 1a — Jones & Klenow: Lambda/BIP (Querschnitt 2007)

Das Blatt **`Levels in 2007`** aus `BeyondGDP500.xls` enthält je Land die konsumäquivalente
Wohlfahrt **λ** (`Welfare (lambda)`), das **BIP pro Kopf y** (`GDP p.c. (y)`, beide USA = 100)
sowie **`log(Ratio)` = ln(λ/y)** — genau das gesuchte Verhältnis Lambda/BIP, bereits berechnet.
Die Quelle ist ein reiner **Querschnitt für 2007** und enthält **keine ISO3-Codes**; die Namen
werden über ein Mapping auf unsere Länderliste abgebildet.

In [2]:
# Datenblock beginnt nach 9 Kopfzeilen; Spalten 0..3 = Name, lambda, y, log(λ/y)
jk_raw = pd.read_excel(JK_PATH, sheet_name='Levels in 2007', header=None, skiprows=9)
jk_raw = jk_raw[jk_raw[0].notna()][[0, 1, 2, 3]].copy()
jk_raw.columns = ['JK_Name', 'lambda', 'y', 'log_lambda_y']
jk_raw['JK_Name'] = jk_raw['JK_Name'].astype(str).str.strip()

# Namensvarianten der J&K-Tabelle -> ISO3 (übrige Namen matchen direkt via NAME_TO_ISO3).
# 'Congo' = Republik Kongo (COG): über y = 4.8 % des US-Niveaus verifiziert (DR Kongo läge <1 %).
JK_NAME_TO_ISO3 = {
    'Bosnia/Herz.': 'BIH', 'Congo': 'COG',       'Czech Rep.':    'CZE',
    'Dominican Rep': 'DOM', 'Egypt': 'EGY',        'Hong Kong':     'HKG',
    'Iran': 'IRN',          'Kyrgyzstan': 'KGZ',   'Macedonia':     'MKD',
    'Russian Fed.': 'RUS',  'Slovakia': 'SVK',     'South Korea':   'KOR',
    'Turkey': 'TUR',        'Venezuela': 'VEN',    'Vietnam':       'VNM',
    'Yemen': 'YEM',
}

def jk_to_iso3(name):
    return NAME_TO_ISO3.get(name) or JK_NAME_TO_ISO3.get(name)

jk_raw['ISO3'] = jk_raw['JK_Name'].map(jk_to_iso3)

# lambda/BIP als Rohquotient (λ und y sind beide zu USA = 100 indexiert) + log-Form
jk = jk_raw.dropna(subset=['ISO3']).copy()
jk['lambda_bip']     = jk['lambda'] / jk['y']
jk['log_lambda_bip'] = jk['log_lambda_y']
jk = jk[jk['ISO3'].isin(INCLUDED_COUNTRIES)][['ISO3', 'lambda', 'y', 'lambda_bip', 'log_lambda_bip']]
jk = jk.sort_values('ISO3').reset_index(drop=True)

fehlt_jk = sorted(set(INCLUDED_COUNTRIES) - set(jk['ISO3']))
print(f'J&K-Werte vorhanden für {len(jk)} von {N_COUNTRIES} Ländern.')
print(f'Ohne Lambda/BIP ({len(fehlt_jk)}): {fehlt_jk}')
jk.head()

J&K-Werte vorhanden für 113 von 119 Ländern.
Ohne Lambda/BIP (6): ['AFG', 'ARE', 'NIC', 'PSE', 'ROU', 'SLV']


,ISO3,lambda,y,lambda_bip,log_lambda_bip
0,ALB,16.8,13.7,1.226277,0.203
1,ARG,21.8,26.2,0.832061,-0.181
2,ARM,12.0,12.3,0.975610,-0.021
3,AUS,90.7,82.1,1.104750,0.099
4,AUT,85.5,80.8,1.058168,0.057


### 1b — IMF WEO-Klassifikation (ein Tab je Jahr)

Die Datei `WEO country classification.xlsx` enthält einen **Referenz-Tab `ISO3`** (unsere 119
Länder mit Namen) sowie je Jahr **2007–2022 einen Tab** (Tab-Name = Jahr) im Format
`ISO3 | Name | Group`. Die Jahres-Tabs listen alle Länder der Welt; wir beschränken auf die
Studienländer. Die Gruppen werden auf **`Advanced`** bzw. **`EMDE`** normalisiert.

In [3]:
weo_xl    = pd.ExcelFile(WEO_PATH)
year_tabs = [s for s in weo_xl.sheet_names if s != 'ISO3']
assert [int(t) for t in year_tabs] == list(range(YEAR_START, YEAR_END + 1)), 'Unerwartete Jahres-Tabs'

GROUP_MAP = {
    'Advanced Economy': 'Advanced',
    'Emerging and Developing Economies': 'EMDE',
}

frames = []
for tab in year_tabs:
    d = weo_xl.parse(tab, header=0)
    d = d.iloc[:, :3].copy()
    d.columns = ['ISO3', 'Name', 'WEO_Group_raw']
    d['ISO3'] = d['ISO3'].astype(str).str.strip()
    d['WEO_Group_raw'] = d['WEO_Group_raw'].astype(str).str.strip()
    d = d[d['ISO3'].isin(INCLUDED_COUNTRIES)]
    d['Year'] = int(tab)
    frames.append(d[['ISO3', 'Year', 'WEO_Group_raw']])

weo_long = pd.concat(frames, ignore_index=True)
weo_long['WEO_Gruppe'] = weo_long['WEO_Group_raw'].map(GROUP_MAP)

# Kontrolle: alle vorkommenden Gruppenbezeichnungen wurden erkannt
unbekannt = sorted(set(weo_long.loc[weo_long['WEO_Gruppe'].isna(), 'WEO_Group_raw']))
assert not unbekannt, f'Unbekannte Gruppenlabels: {unbekannt}'

print(f'WEO-Beobachtungen (Studienländer): {len(weo_long)}')
print('Gruppengrößen je Jahr (Advanced / EMDE):')
print(weo_long.pivot_table(index='Year', columns='WEO_Gruppe', values='ISO3',
                           aggfunc='count', fill_value=0))

WEO-Beobachtungen (Studienländer): 1879


Gruppengrößen je Jahr (Advanced / EMDE):
WEO_Gruppe  Advanced  EMDE
Year                      
2007              27    86
2008              27    85
2009              29    89
2010              29    89
2011              30    88
2012              30    88
2013              30    88
2014              31    87
2015              32    86
2016              32    86
2017              32    86
2018              32    86
2019              32    86
2020              32    86
2021              32    87
2022              32    87


## 2 — Abdeckungs-Übersicht der betrachteten Länder

Je Studienland: liegt ein **Lambda/BIP-Wert** vor, liegen **WEO-Daten** vor (mind. ein Jahr),
und für **wie viele Jahre** (von 16) ein WEO-Wert existiert.

In [4]:
weo_jahre_pro_land = weo_long.dropna(subset=['WEO_Gruppe']).groupby('ISO3')['Year'].nunique()

cov = pd.DataFrame({'ISO3': INCLUDED_COUNTRIES})
cov['Country_Name']         = cov['ISO3'].map(ISO3_TO_NAME)
cov['lambda_bip_vorhanden'] = cov['ISO3'].isin(set(jk['ISO3']))
cov['weo_n_jahre']          = cov['ISO3'].map(weo_jahre_pro_land).fillna(0).astype(int)
cov['weo_vorhanden']        = cov['weo_n_jahre'] > 0
cov['weo_vollstaendig']     = cov['weo_n_jahre'] == N_YEARS
cov = cov.sort_values('ISO3').reset_index(drop=True)

print(f'Länder gesamt:                 {len(cov)}')
print(f'  mit Lambda/BIP:              {cov["lambda_bip_vorhanden"].sum()}')
print(f'  ohne Lambda/BIP:             {(~cov["lambda_bip_vorhanden"]).sum()}')
print(f'  mit WEO (>= 1 Jahr):         {cov["weo_vorhanden"].sum()}')
print(f'  mit WEO für alle 16 Jahre:   {cov["weo_vollstaendig"].sum()}')
print(f'  mit WEO, aber < 16 Jahre:    {(cov["weo_vorhanden"] & ~cov["weo_vollstaendig"]).sum()}')
cov.head(10)

Länder gesamt:                 119
  mit Lambda/BIP:              113
  ohne Lambda/BIP:             6
  mit WEO (>= 1 Jahr):         119
  mit WEO für alle 16 Jahre:   112
  mit WEO, aber < 16 Jahre:    7


,ISO3,Country_Name,lambda_bip_vorhanden,weo_n_jahre,weo_vorhanden,weo_vollstaendig
0,AFG,Afghanistan,False,14,True,False
1,ALB,Albania,True,16,True,True
2,ARE,United Arab Emirates,False,16,True,True
3,ARG,Argentina,True,16,True,True
4,ARM,Armenia,True,16,True,True
5,AUS,Australia,True,16,True,True
6,AUT,Austria,True,16,True,True
7,AZE,Azerbaijan,True,16,True,True
8,BEL,Belgium,True,16,True,True
9,BEN,Benin,True,16,True,True


Länder mit **unvollständiger** WEO-Abdeckung bzw. **ohne** Lambda/BIP:

In [5]:
print('WEO < 16 Jahre:')
display(cov[~cov['weo_vollstaendig']][['ISO3', 'Country_Name', 'weo_n_jahre']]
        .sort_values('weo_n_jahre').reset_index(drop=True))
print('\nOhne Lambda/BIP:')
display(cov[~cov['lambda_bip_vorhanden']][['ISO3', 'Country_Name']].reset_index(drop=True))

WEO < 16 Jahre:


,ISO3,Country_Name,weo_n_jahre
0,PSE,West Bank and Gaza,2
1,AFG,Afghanistan,14
2,IRQ,Iraq,14
3,BIH,Bosnia and Herzegovina,14
4,MNE,Montenegro,14
5,SRB,Serbia,14
6,ZWE,Zimbabwe,15



Ohne Lambda/BIP:


,ISO3,Country_Name
0,AFG,Afghanistan
1,ARE,United Arab Emirates
2,NIC,Nicaragua
3,PSE,West Bank and Gaza
4,ROU,Romania
5,SLV,El Salvador


### Datenqualitäts-Diagnose: WEO-Gruppenwechsel über die Zeit

Zur Plausibilitätskontrolle: welche Länder wechseln im Betrachtungszeitraum die IMF-Gruppe?
Erwartet werden **Aufstiege** in die Advanced-Gruppe (z. B. baltische Staaten, Zypern, Malta,
Tschechien, Slowakei, Slowenien).

> ⚠️ **Auffälligkeit (unverändert aus der Quelle übernommen):** **China (CHN)** ist im Tab **2007**
> als *Advanced Economy* geführt, in allen übrigen Jahren als *EMDE*. Der IMF stuft China in keinem
> WEO-Vintage als Advanced Economy ein — der 2007er-Eintrag ist mit hoher Wahrscheinlichkeit ein
> **Erfassungsfehler** in der Quelldatei (CHN steht alphabetisch direkt hinter CAN). Ich habe die
> Quelldatei **nicht** verändert. Bitte entscheiden, ob der Wert in der Quelle korrigiert oder in
> einem späteren Schritt behandelt werden soll.

In [6]:
chg = (weo_long.dropna(subset=['WEO_Gruppe']).sort_values(['ISO3', 'Year'])
       .groupby('ISO3')['WEO_Gruppe'].agg(anzahl_gruppen='nunique',
                                          erste='first', letzte='last'))
wechsler = chg[chg['anzahl_gruppen'] > 1].copy()
wechsler['Country_Name'] = wechsler.index.map(ISO3_TO_NAME)
print(f'Länder mit Gruppenwechsel im Zeitraum: {len(wechsler)}')
display(wechsler[['Country_Name', 'erste', 'letzte']])

# Für jedes Wechslerland: Jahr(e) des Wechsels sichtbar machen
for iso in wechsler.index:
    reihe = (weo_long[weo_long['ISO3'] == iso].sort_values('Year')
             .set_index('Year')['WEO_Gruppe'])
    wechseljahre = reihe[reihe != reihe.shift()].index.tolist()[1:]
    print(f'{iso} ({ISO3_TO_NAME[iso]}): Wechsel in {wechseljahre} | '
          f'{reihe.iloc[0]} -> {reihe.iloc[-1]}')

Länder mit Gruppenwechsel im Zeitraum: 7


,Country_Name,erste,letzte
ISO3,,,
CHN,China,Advanced,EMDE
CZE,Czechia,EMDE,Advanced
EST,Estonia,EMDE,Advanced
LTU,Lithuania,EMDE,Advanced
LVA,Latvia,EMDE,Advanced
MLT,Malta,EMDE,Advanced
SVK,Slovak Republic,EMDE,Advanced


CHN (China): Wechsel in [2008] | Advanced -> EMDE
CZE (Czechia): Wechsel in [2009] | EMDE -> Advanced
EST (Estonia): Wechsel in [2011] | EMDE -> Advanced
LTU (Lithuania): Wechsel in [2015] | EMDE -> Advanced
LVA (Latvia): Wechsel in [2014] | EMDE -> Advanced
MLT (Malta): Wechsel in [2008] | EMDE -> Advanced
SVK (Slovak Republic): Wechsel in [2009] | EMDE -> Advanced


## 3 — Export der Übersicht

Die Abdeckungs-Übersicht wird als Excel mit zwei Blättern gespeichert:
`Coverage_je_Land` (je Land) und `Zusammenfassung` (Kennzahlen).
Ablage: `code/Created Files for Analysis/classification_coverage_overview.xlsx`.

In [7]:
OUT_XLSX = ANALYSIS_DIR / 'classification_coverage_overview.xlsx'

fehlt_jk_str = ', '.join(sorted(set(INCLUDED_COUNTRIES) - set(jk['ISO3'])))
weo_teil = cov[cov['weo_vorhanden'] & ~cov['weo_vollstaendig']]
weo_teil_str = ', '.join(f"{r.ISO3}({r.weo_n_jahre})" for r in weo_teil.itertuples())

zusammenfassung = pd.DataFrame({
    'Kennzahl': [
        'Studienländer gesamt',
        'mit Lambda/BIP (Jones & Klenow)',
        'ohne Lambda/BIP',
        'ohne Lambda/BIP (Länder)',
        'mit WEO-Daten (>= 1 Jahr)',
        'mit WEO-Daten für alle 16 Jahre',
        'mit WEO-Daten, aber < 16 Jahre',
        'WEO unvollständig (Land/Jahre)',
    ],
    'Wert': [
        len(cov),
        int(cov['lambda_bip_vorhanden'].sum()),
        int((~cov['lambda_bip_vorhanden']).sum()),
        fehlt_jk_str,
        int(cov['weo_vorhanden'].sum()),
        int(cov['weo_vollstaendig'].sum()),
        int((cov['weo_vorhanden'] & ~cov['weo_vollstaendig']).sum()),
        weo_teil_str,
    ],
})

with pd.ExcelWriter(OUT_XLSX, engine='openpyxl') as writer:
    cov.to_excel(writer, sheet_name='Coverage_je_Land', index=False)
    zusammenfassung.to_excel(writer, sheet_name='Zusammenfassung', index=False)

print(f'Gespeichert: {OUT_XLSX.resolve()}')
print(f'  Blatt 1 "Coverage_je_Land":  {len(cov)} Zeilen')
print(f'  Blatt 2 "Zusammenfassung":   {len(zusammenfassung)} Zeilen')

Gespeichert: C:\Users\maier\OneDrive\Dokumente\_Studium\Bachelorarbeit\Practical Analysis\Explaining-Well-Being\code\Created Files for Analysis\classification_coverage_overview.xlsx
  Blatt 1 "Coverage_je_Land":  119 Zeilen
  Blatt 2 "Zusammenfassung":   8 Zeilen


## Zusammenfassung & nächste Schritte

Beide Datenquellen sind eingelesen und auf die 119 Studienländer abgeglichen. Die
Abdeckungs-Übersicht liegt als `classification_coverage_overview.xlsx` vor.

**Kernbefunde der Abdeckung:**
- **Lambda/BIP (Jones & Klenow):** für **113** der 119 Länder vorhanden; es fehlen **6**
  (AFG, ARE, NIC, PSE, ROU, SLV). Quelle ist ein Querschnitt 2007 → Wert ist zeitkonstant.
- **WEO-Klassifikation:** für **alle 119** Länder mind. ein Jahr vorhanden; **112** haben alle
  16 Jahre. Unvollständig sind v. a. PSE (nur 2 Jahre) sowie AFG, BIH, IRQ, MNE, SRB (14) und
  ZWE (15).
- **Datenqualität:** CHN ist 2007 fälschlich als *Advanced* geführt (siehe Diagnose oben).

**Noch offen (spätere Schritte):** Konstruktion der vier Merkmale — (1) IMF-Gruppe je Land×Jahr,
(2) Lambda/BIP als Feature, (3) global über-/unterdurchschnittlich, (4) über-/unterdurchschnittlich
innerhalb der IMF-Gruppe je Jahr — sowie deren Anfügen als Metadaten-Spalten an die Analyse-DB.